## TO GET QUESTIONS FOR THE SOLUTIONS OF STUDENTS FROM QB

In [4]:
import sys
sys.path.append('/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading')
from ui.z2_ocr.Physics.ocr_handler import OCRHandler
import pandas as pd
import os
from typing import Optional
from pathlib import Path

# Assuming extract_pdf_name_from_url function and TextPreprocessor class are defined elsewhere
from utils import extract_pdf_name_from_url
from data_processing.preprocessor import TextPreprocessor
import ocr_handler

def load_metadata(hw_solution_with_qb_meta_csv: str) -> pd.DataFrame:
    """
    Load metadata from CSV file.

    Args:
        hw_solution_with_qb_meta_csv: Path to the metadata CSV file.

    Returns:
        pd.DataFrame: DataFrame containing metadata.
    """
    try:
        # Load CSV into a DataFrame
        qb_meta_df = pd.read_csv(hw_solution_with_qb_meta_csv, low_memory=False)
        print(f"✓ Metadata loaded from {hw_solution_with_qb_meta_csv}")
        return qb_meta_df
    except FileNotFoundError:
        print(f"⚠️ Metadata CSV file not found: {hw_solution_with_qb_meta_csv}")
        return pd.DataFrame()
    except Exception as e:
        print(f"⚠️ Error loading metadata: {str(e)}")
        return pd.DataFrame()

def clean_solutions(qb_meta_df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the solutions for the questions using TextPreprocessor.

    Args:
        qb_meta_df: DataFrame containing the metadata with `textsolutions`.

    Returns:
        pd.DataFrame: Updated DataFrame with cleaned solutions.
    """
    try:
        # Initialize TextPreprocessor for cleaning solutions
        text_preprocessor = TextPreprocessor()
        qb_meta_df['cleaned_solution'] = qb_meta_df['textsolutions'].apply(text_preprocessor.process_solution)
        print("✓ Successfully added cleaned_solution column")
    except Exception as e:
        print(f"⚠️ Could not add cleaned_solution column: {e}")
        qb_meta_df['cleaned_solution'] = ""  # Adding empty cleaned_solution column in case of failure

    return qb_meta_df
def clean_questions(qb_meta_df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the questions using TextPreprocessor.

    Args:
        qb_meta_df: DataFrame containing the metadata with `content` (questions).

    Returns:
        pd.DataFrame: Updated DataFrame with cleaned questions.
    """
    try:
        # Initialize TextPreprocessor for cleaning questions (same as for solutions)
        text_preprocessor = TextPreprocessor()
        
        # Apply process_solution (or create a process_question if needed) to clean questions
        qb_meta_df['cleaned_question'] = qb_meta_df['content'].apply(text_preprocessor.process_solution)
        print("✓ Successfully added cleaned_question column")
    except Exception as e:
        print(f"⚠️ Could not add cleaned_question column: {e}")
        qb_meta_df['cleaned_question'] = ""  # Adding empty cleaned_question column in case of failure

    return qb_meta_df


def extract_pdf_names(qb_meta_df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract PDF names from the 'UPLOADED_ANS' column in the metadata CSV.

    Args:
        qb_meta_df: DataFrame containing the metadata.

    Returns:
        pd.DataFrame: Updated DataFrame with an added 'pdf_name' column.
    """
    if 'UPLOADED_ANS' in qb_meta_df.columns:
        qb_meta_df['pdf_name'] = qb_meta_df['UPLOADED_ANS'].apply(extract_pdf_name_from_url)
        print("✓ Extracted PDF names from 'UPLOADED_ANS' column")
    else:
        print("⚠️ 'UPLOADED_ANS' column not found in metadata CSV")

    return qb_meta_df

def get_pdf_metadata(qb_meta_df: pd.DataFrame, pdf_name: str) -> Optional[pd.DataFrame]:
    """
    Get metadata for a specific PDF based on its name.

    Args:
        qb_meta_df: DataFrame containing the metadata.
        pdf_name: The name of the PDF to filter metadata.

    Returns:
        pd.DataFrame: Filtered DataFrame for the specified PDF, or None if not found.
    """
    pdf_metadata = qb_meta_df[qb_meta_df['pdf_name'] == pdf_name]
    if pdf_metadata.empty:
        print(f"⚠️ No metadata found for PDF: {pdf_name}")
        return None
    return pdf_metadata

def generate_question_list(test_df: pd.DataFrame, prompt_version: str) -> list:
    ocr_handler_instance = OCRHandler()
    image_dir = "./images"
    os.makedirs(image_dir, exist_ok=True)
    if prompt_version == 'v12':
        questions_list = ocr_handler_instance.get_interleaved_question_solution_list(test_df, image_dir)
        print(f"✓ Generated interleaved question-solution list for QB-guided assessment")
    else:
        questions_list = ocr_handler_instance.get_question_list(test_df, image_dir)
        print(f"✓ Generated question list")
    return questions_list

def process_pdf(hw_solution_with_qb_meta_csv: str, pdf_name: str, prompt_version: str):
    """
    Main method to process a specific PDF's metadata, clean solutions, and generate a question list.

    Args:
        hw_solution_with_qb_meta_csv: Path to the metadata CSV file.
        pdf_name: Name of the PDF for which to process metadata.
        prompt_version: The prompt version to generate the question list ('v12' or other versions).
    """
    qb_meta_df = load_metadata(hw_solution_with_qb_meta_csv)
    if qb_meta_df.empty:
        return
    
    # Extract PDF names from URL (if UPLOADED_ANS exists)
    qb_meta_df = extract_pdf_names(qb_meta_df)
    
    # Clean solutions if possible
    qb_meta_df = clean_solutions(qb_meta_df)
    
    # Clean questions as well
    qb_meta_df = clean_questions(qb_meta_df)  # Add this line to clean questions

    # Get metadata for the specified PDF
    pdf_metadata = get_pdf_metadata(qb_meta_df, pdf_name)
    if pdf_metadata is None:
        return

    # Prepare DataFrame for OCR processing
    test_df = pdf_metadata[['content', 'Question_no', 'Marks', 'QB_ID', 'textsolutions', 'streams', 'oldtags', 'cleaned_solution', 'cleaned_question']].sort_values(by='Question_no')

    # Generate the question list
    questions_list = generate_question_list(test_df, prompt_version)

    return questions_list

if __name__ == "__main__":
    # Example usage
    hw_solution_with_qb_meta_csv = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/ui/z2_ocr/Physics/hw_df_with_solutions_and_questions.csv"
    pdf_name = "10021040891039611111693747018.pdf"  # Example PDF name
    prompt_version = "v12"  # Example prompt version

    questions_list = process_pdf(hw_solution_with_qb_meta_csv, pdf_name, prompt_version)
    if questions_list:
        print(f"Generated {len(questions_list)} questions.")
        # Clean and save questions_list
        from bs4 import BeautifulSoup
        import json

        def clean_html(raw_html):
            return BeautifulSoup(raw_html, "html.parser").get_text(separator=" ", strip=True)

        def extract_and_clean(item):
            try:
                data = json.loads(item)
                if isinstance(data, list) and "questionStem" in data[0]:
                    question_html = data[0]["questionStem"]["text"]
                    return clean_html(question_html)
                else:
                    return clean_html(item)
            except Exception:
                return clean_html(item)

        with open("questions_list_cleaned.txt", "w", encoding="utf-8") as f:
            for item in questions_list:
                if isinstance(item, str):
                    cleaned = extract_and_clean(item)
                    f.write(cleaned + "\n")
                else:
                    f.write(str(item) + "\n")
        print("Cleaned questions list saved to questions_list_cleaned.txt")




✓ Metadata loaded from /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/ui/z2_ocr/Physics/hw_df_with_solutions_and_questions.csv
✓ Extracted PDF names from 'UPLOADED_ANS' column


/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/ui/z2_ocr/Physics/data_processing/preprocessor.py:22: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text, 'html.parser')


✓ Successfully added cleaned_solution column
✓ Successfully added cleaned_question column
Initializing OCR Handler...
✓ Successfully imported ocr_with_questions
✓ Successfully imported extract_text_from_html
✓ Successfully imported visualize_and_save_question
✓ Successfully imported resize_image
OCR Handler availability: True
✓ Generated interleaved question-solution list for QB-guided assessment
Generated 33 questions.
Cleaned questions list saved to questions_list_cleaned.txt


In [1]:
import sys
sys.path.append('/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading')
from ui.z2_ocr.Physics.ocr_handler import OCRHandler
import pandas as pd
import os
from typing import Optional
from pathlib import Path
import re

# Set the PDF directory
pdf_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/phy"  # <-- Set this to your PDF directory
pdf_files = [f for f in os.listdir(pdf_dir) if f.lower().endswith('.pdf')]  # Get all PDF files in the directory

# Assuming extract_pdf_name_from_url function and TextPreprocessor class are defined elsewhere
from utils import extract_pdf_name_from_url
from data_processing.preprocessor import TextPreprocessor
import ocr_handler

def load_metadata(hw_solution_with_qb_meta_csv: str) -> pd.DataFrame:
    """
    Load metadata from CSV file.

    Args:
        hw_solution_with_qb_meta_csv: Path to the metadata CSV file.

    Returns:
        pd.DataFrame: DataFrame containing metadata.
    """
    try:
        qb_meta_df = pd.read_csv(hw_solution_with_qb_meta_csv, low_memory=False)
        print(f"✓ Metadata loaded from {hw_solution_with_qb_meta_csv}")
        return qb_meta_df
    except FileNotFoundError:
        print(f"⚠️ Metadata CSV file not found: {hw_solution_with_qb_meta_csv}")
        return pd.DataFrame()
    except Exception as e:
        print(f"⚠️ Error loading metadata: {str(e)}")
        return pd.DataFrame()

def remove_angle_brackets(text):
    # Remove anything between < and >, including the brackets
    return re.sub(r'<.*?>', '', text)

def clean_text(qb_meta_df: pd.DataFrame, column_name: str, new_column_name: str) -> pd.DataFrame:
    """
    Clean text (either solutions or questions) using TextPreprocessor.

    Args:
        qb_meta_df: DataFrame containing the metadata.
        column_name: The column name of the text to be cleaned (e.g., 'content' or 'textsolutions').
        new_column_name: The name of the new column to store cleaned text (e.g., 'cleaned_solution' or 'cleaned_question').

    Returns:
        pd.DataFrame: Updated DataFrame with cleaned text in the new column.
    """
    try:
        text_preprocessor = TextPreprocessor()
        qb_meta_df[new_column_name] = qb_meta_df[column_name].apply(text_preprocessor.process_solution)
        print(f"✓ Successfully added {new_column_name} column")
    except Exception as e:
        print(f"⚠️ Could not add {new_column_name} column: {e}")
        qb_meta_df[new_column_name] = ""  # Adding empty column in case of failure

    return qb_meta_df

def extract_pdf_names(qb_meta_df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract PDF names from the 'UPLOADED_ANS' column in the metadata CSV.

    Args:
        qb_meta_df: DataFrame containing the metadata.

    Returns:
        pd.DataFrame: Updated DataFrame with an added 'pdf_name' column.
    """
    if 'UPLOADED_ANS' in qb_meta_df.columns:
        qb_meta_df['pdf_name'] = qb_meta_df['UPLOADED_ANS'].apply(extract_pdf_name_from_url)
        print("✓ Extracted PDF names from 'UPLOADED_ANS' column")
    else:
        print("⚠️ 'UPLOADED_ANS' column not found in metadata CSV")

    return qb_meta_df

def get_pdf_metadata(qb_meta_df: pd.DataFrame, pdf_name: str) -> Optional[pd.DataFrame]:
    """
    Get metadata for a specific PDF based on its name.

    Args:
        qb_meta_df: DataFrame containing the metadata.
        pdf_name: The name of the PDF to filter metadata.

    Returns:
        pd.DataFrame: Filtered DataFrame for the specified PDF, or None if not found.
    """
    pdf_metadata = qb_meta_df[qb_meta_df['pdf_name'] == pdf_name]
    if pdf_metadata.empty:
        print(f"⚠️ No metadata found for PDF: {pdf_name}")
        return None
    return pdf_metadata

def generate_question_list(test_df: pd.DataFrame, prompt_version: str) -> list:
    ocr_handler_instance = OCRHandler()
    image_dir = "./images"
    os.makedirs(image_dir, exist_ok=True)
    if prompt_version == 'v12':
        questions_list = ocr_handler_instance.get_interleaved_question_solution_list(test_df, image_dir)
        print(f"✓ Generated interleaved question-solution list for QB-guided assessment")
    else:
        questions_list = ocr_handler_instance.get_question_list(test_df, image_dir)
        print(f"✓ Generated question list")
    return questions_list

def process_pdf(hw_solution_with_qb_meta_csv: str, pdf_name: str, prompt_version: str):
    """
    Main method to process a specific PDF's metadata, clean solutions, and generate a question list.

    Args:
        hw_solution_with_qb_meta_csv: Path to the metadata CSV file.
        pdf_name: Name of the PDF for which to process metadata.
        prompt_version: The prompt version to generate the question list ('v12' or other versions).
    """
    qb_meta_df = load_metadata(hw_solution_with_qb_meta_csv)
    if qb_meta_df.empty:
        return
    
    # Extract PDF names from URL (if UPLOADED_ANS exists)
    qb_meta_df = extract_pdf_names(qb_meta_df)
    
    # Clean solutions if possible
    qb_meta_df = clean_text(qb_meta_df, 'textsolutions', 'cleaned_solution')
    
    # Clean questions as well
    qb_meta_df = clean_text(qb_meta_df, 'content', 'cleaned_question')  # Clean questions using the same method

    # Get metadata for the specified PDF
    pdf_metadata = get_pdf_metadata(qb_meta_df, pdf_name)
    if pdf_metadata is None:
        return

    # Prepare DataFrame for OCR processing
    test_df = pdf_metadata[['content', 'Question_no', 'Marks', 'QB_ID', 'textsolutions', 'streams', 'oldtags', 'cleaned_solution', 'cleaned_question']].sort_values(by='Question_no')

    # Generate the question list
    questions_list = generate_question_list(test_df, prompt_version)

    return questions_list

def process_all_pdfs(hw_solution_with_qb_meta_csv: str, prompt_version: str):
    """
    Process all PDFs in the directory.

    Args:
        hw_solution_with_qb_meta_csv: Path to the metadata CSV file.
        prompt_version: The prompt version to generate the question list ('v12' or other versions).
    """
    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}...")
        questions_list = process_pdf(hw_solution_with_qb_meta_csv, pdf_file, prompt_version)
        if questions_list:
            print(f"Generated {len(questions_list)} questions for {pdf_file}.")
            # Clean and save questions_list
            from bs4 import BeautifulSoup
            import json

            def clean_html(raw_html):
                return BeautifulSoup(raw_html, "html.parser").get_text(separator=" ", strip=True)

            def extract_and_clean(item):
                try:
                    data = json.loads(item)
                    if isinstance(data, list) and "questionStem" in data[0]:
                        question_html = data[0]["questionStem"]["text"]
                        return clean_html(question_html)
                    else:
                        return clean_html(item)
                except Exception:
                    return clean_html(item)

            output_folder = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/qb_txt"
            os.makedirs(output_folder, exist_ok=True)
            txt_filename = os.path.splitext(pdf_file)[0] + ".txt"
            output_path = os.path.join(output_folder, txt_filename)

            with open(output_path, "w", encoding="utf-8") as f:
                for item in questions_list:
                    if isinstance(item, str):
                        cleaned = extract_and_clean(item)
                        cleaned = remove_angle_brackets(cleaned)  # Remove <...>
                        if not cleaned.strip().startswith("QB Solution"):
                          f.write(cleaned + "\n")
                    else:
                         pass  # Ignore non-string items

            print(f"Cleaned questions list saved to {output_path}")

# === Example Usage ===
if __name__ == "__main__":
    hw_solution_with_qb_meta_csv = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/ui/z2_ocr/Physics/hw_df_with_solutions_and_questions.csv"
    prompt_version = "v12"  # Example prompt version

    # Process all PDFs in the PDF directory
    process_all_pdfs(hw_solution_with_qb_meta_csv, prompt_version)


Initializing OCR Handler...
✓ Successfully imported ocr_with_questions
✓ Successfully imported extract_text_from_html
✓ Successfully imported visualize_and_save_question
✓ Successfully imported resize_image
OCR Handler availability: True
Initializing OCR Handler...
✓ Successfully imported ocr_with_questions
✓ Successfully imported extract_text_from_html
✓ Successfully imported visualize_and_save_question
✓ Successfully imported resize_image
OCR Handler availability: True
Initializing OCR Handler...
✓ Successfully imported ocr_with_questions
✓ Successfully imported extract_text_from_html
✓ Successfully imported visualize_and_save_question
✓ Successfully imported resize_image
OCR Handler availability: True
Processing 1002110109818371111684651004.pdf...
✓ Metadata loaded from /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/ui/z2_ocr/Physics/hw_df_with_solutions_and_questions.csv
✓ Extracted PDF names from 'UPLOADED_ANS' column


/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/data_processing/preprocessor.py:22: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text, 'html.parser')


✓ Successfully added cleaned_solution column
✓ Successfully added cleaned_question column
Initializing OCR Handler...
✓ Successfully imported ocr_with_questions
✓ Successfully imported extract_text_from_html
✓ Successfully imported visualize_and_save_question
✓ Successfully imported resize_image
OCR Handler availability: True
✓ Generated interleaved question-solution list for QB-guided assessment
Generated 12 questions for 1002110109818371111684651004.pdf.
Cleaned questions list saved to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/qb_txt/1002110109818371111684651004.txt
Processing 10021105101083421111694960631.pdf...
✓ Metadata loaded from /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/ui/z2_ocr/Physics/hw_df_with_solutions_and_questions.csv
✓ Extracted PDF names from 'UPLOADED_ANS' column


/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/data_processing/preprocessor.py:22: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text, 'html.parser')


KeyboardInterrupt: 

## send questions qb and images of studentsolutions to get the ocr from gemini - PREDICTIONS

In [ ]:
import os
from dotenv import load_dotenv
import google.generativeai as genai
import fitz  # PyMuPDF for PDF processing
from PIL import Image
import json
import base64
import cv2
import time
import numpy as np
import base64


# === Load API Key ===
load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_GEMINI_API"))

# Set model
model_name = "gemini-2.5-pro"
model = genai.GenerativeModel(model_name)

# === Prompt for Gemini ===
PROMPT = """

**Overall Goal**: Process a Question Bank (text) and handwritten solutions to create a structured JSON output that accurately associates questions with their transcribed handwritten solutions, ensuring a **verbatim copy** of all original content and formatting from the handwritten source.

---

## 1. System Instruction & Role

**Role**: You are a meticulous digital archivist specialized in **verbatim transcription** and collation of academic materials. You have access to a Question Bank (QB) provided as plain text and **pre-processed OCR results (raw transcribed text and detected diagrams)** from corresponding handwritten student answer sheets (images). Your paramount task is to create a perfect, structured digital copy of the student's work by accurately associating each solution with its respective question, ensuring **absolute fidelity to the handwritten source**.

**Core Task**:
Your goal is to:
1.  **Match each solution from the PROVIDED OCR data to the corresponding question** from the Question Bank.
2.  **Transcribe and preserve ALL content (text, numbers, symbols, and formatting) EXACTLY as it appears** in the raw OCR results of the handwritten images.
3.  **Create a structured and accurate digital record of each question's solution** using **only the raw, transcribed content from the handwritten student answers**.

---

## 2. Input Data Context

You will be provided with the following content as input:

*   **`question_bank_content` (String)**: The full text content of the Question Bank, where each question is clearly numbered.
*   **`solution_ocr_data` (JSON Object)**: A structured representation of the OCR output from the handwritten solution pages. This data **already contains the raw transcribed text** for various segments, along with bounding box information for text and detected diagrams. This is your **sole source for the `ocr_text` field**.

---

## 3. Critical Rules & Constraints

**Absolute Adherence**: Failure to follow any rule will result in an incorrect output.

1.  **Transcription Fidelity - VERBATIM Copy**:
    *   The `ocr_text` field for each solution MUST be a **VERBATIM transcription of the handwritten content** from the provided OCR data.
    *   **Every character, including all spacing, punctuation, line breaks (represented by `\n`), and any numerical or mathematical expressions, MUST be preserved exactly as it appears in the provided OCR data.**
    *   **DO NOT** correct any spelling, grammatical, or punctuation errors. Preserve apparent mistakes or inconsistencies (e.g., If "metabolities" is written instead of "metabolites", it **must** remain unchanged).
    *   **CRITICAL: DO NOT add, infer, paraphrase, summarize, or extrapolate ANY information not explicitly present in the original handwritten content as provided in the OCR data.** This means no adding introductory phrases, descriptive labels (like "Radius of curvature" or "focal length") if the student didn't explicitly write them, or completing abbreviated thoughts.

2.  **Unicode Characters**:
    *   Avoid using escape sequences (e.g., `\u25b3`, `\u00b2`). Use the direct Unicode character (e.g., ∆, ², cm, α, β, ∑, ∫).

3.  **Tables**:
    *   If a table structure is identified in the handwritten answer (via OCR), output the information in a proper Markdown table format within the `ocr_text`.

4.  **Symbol Handling - Preserve All Meaningful Symbols**:
    *   **CRITICAL: Preserve ALL meaningful symbols that are an integral part of the content, structure, or flow of the student's solution.** This includes, but is not limited to: mathematical operators (+, -, *, /, =), logical arrows (e.g., `->` when indicating a derivation, chemical reaction, or flow), unit symbols (e.g., cm, m/s²), Greek letters (e.g., α, β), or any character that contributes to the meaning or structure of the solution.
    *   Only omit purely decorative scribbles or clear OCR errors that are non-textual noise (e.g., stray pixels, pen smudges).

5.  **No External Knowledge/Correction**:
    *   **DO NOT** add any external knowledge, commentary, or context.
    *   **DO NOT** use the Question Bank content to generate or modify the `ocr_text` of the solution. The solution text (`ocr_text`) must come **EXCLUSIVELY** from the raw transcribed content of the handwritten student answers provided in `solution_ocr_data`.

---

## 4. Specific Instructions for Processing Questions and Solutions

1.  **Matching Questions and Solutions**:
    *   Carefully parse the `question_bank_content` to identify each distinct question and its unique question number.
    *   Then, analyze the `solution_ocr_data` to precisely map which segments of the handwritten solutions correspond to which question numbers from the `question_bank_content`.
    *   If a handwritten solution indicates sub-parts (e.g., "11. (a)", "11. (b)"), group these under the main question number (e.g., `question_number: 11`). The `ocr_text` should concatenate the solutions for all sub-parts of a main question in their logical order, preserving line breaks.

2.  **Handling Multiple Solution Parts/Pages**:
    *   If a solution for a single question spans multiple pages or contains multiple distinct parts (e.g., "11. (1)", "11. (2)"), ensure that the entire solution content is consolidated and grouped together under the corresponding main question number in the `ocr_text` field.
    *   The `pages` array should accurately list all unique page numbers on which *any* part of that question's solution appears.

3.  **Handling Diagrams**:
    *   For any diagrams (graphs, illustrations, flowcharts) identified within the `solution_ocr_data` as part of a question's solution, you **must** include them in the `diagrams` array for that question.
    *   **Crucially**: Also include any text *around* the diagram that is part of the student's solution or label for the diagram within the `ocr_text` for that question.

---

## 5. Instructions for Output

-   **The output MUST be a single, valid JSON array `[]`**.
-   Each object `{}` within the array must correspond to a main question and its associated solution(s).

Each object must include the following keys:

1.  `question_number`:
    *   **Type**: Integer.
    *   **Source**: The main question number (e.g., `11`, `12`) as extracted from the `question_bank_content`.
2.  `question_text`:
    *   **Type**: String.
    *   **Source**: The full text of the question as extracted verbatim from the `question_bank_content`.
3.  `ocr_text`:
    *   **Type**: String.
    *   **Source**: The full, raw, and **verbatim transcribed solution** for that question and all its sub-parts (if any), **EXCLUSIVELY from the provided OCR data** of the handwritten solution.
    *   **Content**: Concatenated text from all relevant handwritten pages/sections for this question, strictly adhering to all transcription fidelity rules (no corrections, preserve errors, direct Unicode, proper tables, and **preserve all meaningful symbols and original line breaks**).
4.  `diagrams`:
    *   **Type**: Array of objects.
    *   **Content**: Include this array **only if** diagrams are present in the handwritten solution. If no diagrams, omit the key or provide an empty array `[]`.
    *   Each diagram object within this array must include:
        *   `id`:
            *   **Type**: String.
            *   **Value**: A unique, descriptive identifier for the diagram (e.g., `"diagram_1"`, `"graph_for_q11a"`).
        *   `coordinates`:
            *   **Type**: String.
            *   **Value**: The precise coordinates of the diagram on its original page, in the format `"x_mid,y_mid,width,height"`.
            *   **Format**: All values must be normalized between `0` and `1` (relative to image dimensions, e.g., `0.5,0.5,0.2,0.3`).
        *   `diagram_class`:
            *   **Type**: String.
            *   **Value**: Specify whether the object is a `"graph"` or a `"diagram"`.
        *   `page_number`:
            *   **Type**: Integer.
            *   **Value**: The page number (starting from `1`) on which the diagram appears.
5.  `pages`:
    *   **Type**: Array of integers.
    *   **Content**: A list of all unique page numbers (starting from `1`) on which *any* part of the solution for this specific question appears. The numbers in the array should be sorted in ascending order.

---

## 6. Example Output:

```json
[
  {
    "question_number": 11,
    "question_text": "Describe the process of glycolysis, including its main products and where it occurs in the cell.",
    "ocr_text": "11. (1) Glycolysis is the breakdown of glucose into pyruvate. It occurs in the sytosol.\n(2) The main products are 2 ATP, 2 NADH, and 2 pyruvate molecules.\nThe pathway involves multiple steps, starting with glucose phosphorylation and ending with pyruvate.",
    "diagrams": [
      {
        "id": "diagram_glycolysis_pathway",
        "coordinates": "0.5,0.5,0.2,0.3",
        "diagram_class": "diagram",
        "page_number": 3
      }
    ],
    "pages": [2, 3]
  },
  {
    "question_number": 12,
    "question_text": "State the formula for force and identify the units of its components in the SI system.",
    "ocr_text": "12. The solution involves using the formula: F = ma.\n[NOTE: The solution includes a diagram on page 5.]",
    "diagrams": [
      {
        "id": "diagram_force_vector",
        "coordinates": "0.6,0.6,0.15,0.2",
        "diagram_class": "diagram",
        "page_number": 5
      }
    ],
    "pages": [4, 5]
  }
]
"""

# === Load Base64 Images Function ===
def load_base64_images(folder_path, num_pages, dim=736):  # <-- add dim as an argument
    """
    Load image files from the specified folder and return them as base64-encoded strings.

    Args:
        folder_path (str): The folder containing the image files.
        num_pages (int): The number of pages (images) to process.

    Returns:
        list: A list of base64-encoded image strings.
    """
    b64_list = []
    for i in range(num_pages):
        img_filename_dim = f"DIM_{dim}_PAGE_{i + 1}.jpeg"
        img_filename_default = f"page_{i + 1}.jpeg"

        img_path_dim = os.path.join(folder_path, img_filename_dim)
        img_path_default = os.path.join(folder_path, img_filename_default)

        if os.path.exists(img_path_dim):
            path = img_path_dim
        elif os.path.exists(img_path_default):
            path = img_path_default
        else:
            print(f"❌ Image {img_filename_dim} or {img_filename_default} not found in {folder_path}")
            continue

        with open(path, "rb") as f:
            b64_list.append(base64.b64encode(f.read()).decode())

    return b64_list


# === Resize Image Function ===
def resize_image(image, dim=736, save_path=None):
    image1 = np.array(image.convert('RGB'))  # Ensure the image is in RGB mode
    original_size = image1.shape  # (height, width, channels)
    image1 = image1.mean(axis=2)  # Convert image to grayscale
    h, w = image1.shape
    if w > h:
        new_w = dim
        new_h = int(h * (dim / w))
    else:
        new_h = dim
        new_w = int(w * (dim / h))
    resized_image = cv2.resize(image1, (new_w, new_h), interpolation=cv2.INTER_AREA)
    resized_image_pil = Image.fromarray(resized_image)
    resized_image_pil = resized_image_pil.convert('RGB')  # Convert to RGB before saving
    if save_path:
        resized_image_pil.save(save_path)
    return original_size, (new_h, new_w), resized_image_pil

# === Convert PDF to Images ===
def pdf_to_images(pdf_path, output_folder):
    doc = fitz.open(pdf_path)
    images = []
    num_pages = doc.page_count
    for i in range(num_pages):
        page = doc.load_page(i)
        pix = page.get_pixmap(dpi=300)  # Convert PDF page to image at 300 dpi
        img_path_default = os.path.join(output_folder, f"page_{i + 1}.jpeg")
        pix.save(img_path_default)  # Save image as JPEG
        images.append(img_path_default)  # Add path to images list
    return images, num_pages

# === Load TXT Files ===
def load_txt_file(txt_file_path):
    """
    Load a single TXT file containing questions.

    Args:
        txt_file_path (str): The path to the TXT file.

    Returns:
        str: The content of the TXT file.
    """
    if os.path.exists(txt_file_path):
        with open(txt_file_path, "r", encoding="utf-8") as f:
            return f.read()
    else:
        print(f"❌ Text file {txt_file_path} not found.")
        return ""

# === Function to Send Data to Gemini ===
def send_to_gemini(resized_images_objs, txt_file_content):
    try:
        # Combine the prompt, resized image objects, and the question text into a single request
        response = model.generate_content([PROMPT] + resized_images_objs + [txt_file_content])  # Batch processing
        
        # Parse the response (clean and load as JSON)
        raw = response.text.strip()
        cleaned = raw.strip('```json').strip('```').strip()
        parsed = json.loads(cleaned)
        return parsed
    except Exception as e:
        print(f"❌ Failed to process images: {e}")
        return None

# === Function to Process All PDF and TXT Files ===
def process_all_files(input_pdf_dir, input_txt_dir, output_json_dir):
    # Iterate through all files in the PDF directory
    for root, dirs, files in os.walk(input_pdf_dir):
        for file in files:
            if file.endswith(".pdf"):
                pdf_filename = os.path.splitext(file)[0]
                txt_filename = f"{pdf_filename}.txt"  # Match with the corresponding TXT file
                
                # Check if the corresponding TXT file exists
                txt_file_path = os.path.join(input_txt_dir, txt_filename)
                if os.path.exists(txt_file_path):
                    print(f"Processing {pdf_filename}...")

                    # Define the output folder path using the PDF filename
                    output_folder = os.path.join(output_json_dir, pdf_filename)
                    os.makedirs(output_folder, exist_ok=True)

                    # Process the corresponding PDF and TXT file
                    try:
                        input_pdf_path = os.path.join(root, file)
                        images, num_pages = pdf_to_images(input_pdf_path, output_folder)

                        resized_images = []
                        for page_num in range(len(images)):
                            img_path = images[page_num]
                            original_size, new_size, resized_image = resize_image(Image.open(img_path), dim=736)

                            # Save resized image
                            img_filename_dim = f"DIM_736_PAGE_{page_num + 1}.jpeg"
                            resized_image.save(os.path.join(output_folder, img_filename_dim))

                            resized_images.append(resized_image)

                        # Load the single TXT file containing the questions
                        txt_file_content = load_txt_file(txt_file_path)

                        # Load base64 images and send both images and questions to Gemini
                        images_b64 = load_base64_images(output_folder, len(images))
                        results = send_to_gemini(images_b64, txt_file_content)

                        if results:
                            output_json_filename = f"{pdf_filename}.json"
                            json_path = os.path.join(output_folder, output_json_filename)

                            # Save the OCR results
                            with open(json_path, "w") as f:
                                json.dump(results, f, indent=3)

                            print(f"OCR results saved to {json_path}")
                        else:
                            print("❌ No results from Gemini OCR")

                    except Exception as e:
                        print(f"❌ Error processing {input_pdf_path}: {e}")
                else:
                    print(f"⚠️ Corresponding TXT file not found for {pdf_filename}")

# === Example Usage ===
if __name__ == "__main__":
    input_pdf_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/phy"  # Change this to the correct directory path for PDFs
    input_txt_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/qb_txt"  # Change this to the correct directory path for TXT files
    output_json_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/phy_json"  # Change this to the desired output directory
    
    # Process all PDF and TXT files in the input directories
    process_all_files(input_pdf_dir, input_txt_dir, output_json_dir)


Processing 1002110109818371111684651004...
OCR results saved to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/Physics/Physics_Gemini/phy_json/1002110109818371111684651004/1002110109818371111684651004.json
Processing 10021105101083421111694960631...


KeyboardInterrupt: 